# Thin-Market Liquidity Comparison (Kalshi macro series)

Grounds `paper/sections/3_manipulation_risk.md`. The FEDS paper (Diercks, Katz & Wright 2026) documents that the **fed funds rate** series is extremely liquid -- daily volumes "frequently above a million" and peaking near 100 million contracts around a September FOMC meeting (Figure 3, p.12) -- but gives **no equivalent volume figures** for the thinner series (GDP growth, probability of recession, core CPI-for-year), which have much shorter listing histories (Table 1, p.8: core-CPI-for-year began 2025, probability-of-recession began 2022).

This notebook plots daily liquidity **by series on a common scale**, so the gap between the FFR market the FEDS paper focuses on and the thin series a manipulator would more plausibly target is visible in one chart. That chart is the single most persuasive piece of original evidence Section 3 can add.

**It ships with a synthetic volume generator so it runs end to end with no credentials and no network** (Kalshi's API is external and may be blocked in restricted sandboxes). Swap in a real pull via `src/kalshi_api.py` where flagged.

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(11)

## 1. Table 1 series and their (relative) market thickness

The `thickness` column is a prior, not data -- it encodes the qualitative ordering the FEDS paper implies (FFR hugely liquid; GDP / recession / core-CPI-annual thin and recently listed) and drives the synthetic generator below. Replace the whole synthetic block with a real pull to get actual numbers.

In [ ]:
series = pd.DataFrame([
    # series ticker guesses are illustrative -- verify real tickers via kalshi_api.list_events()
    {"series": "Fed Funds Rate Decision", "ticker": "KXFED",  "first_year": 2022, "theme": "Monetary Policy", "thickness": 1.0},
    {"series": "CPI YoY",                 "ticker": "KXCPIYOY", "first_year": 2022, "theme": "Inflation",       "thickness": 0.35},
    {"series": "Unemployment Rate",       "ticker": "KXUNEMP", "first_year": 2021, "theme": "Labor Market",    "thickness": 0.25},
    {"series": "Core CPI for Year",       "ticker": "KXCORECPIYR", "first_year": 2025, "theme": "Inflation",   "thickness": 0.05},
    {"series": "GDP Growth (Quarterly)",  "ticker": "KXGDP",   "first_year": 2021, "theme": "Growth",         "thickness": 0.06},
    {"series": "Probability of US Recession", "ticker": "KXRECESSION", "first_year": 2022, "theme": "Growth",  "thickness": 0.04},
])
series

## 2. Synthetic daily-volume generator (no network)

Produces ~180 trading days of daily contract volume per series. The FFR series spikes into the millions around FOMC weeks; the thin series stay orders of magnitude smaller. This mimics the *shape* the FEDS paper describes without claiming the exact magnitudes.

In [ ]:
def generate_synthetic_volume(thickness: float, n_days: int = 180, fomc_days=(45, 90, 135)):
    """Daily contract volume for one series. FFR-like (thickness~1) peaks near
    ~1e8 around FOMC weeks; thin series (thickness~0.05) stay near ~1e4-1e5."""
    base = 1e8 * thickness
    days = np.arange(n_days)
    vol = np.full(n_days, base * 0.02)
    for f in fomc_days:
        vol += base * np.exp(-((days - f) ** 2) / (2 * 6.0 ** 2))  # FOMC-week bump
    vol *= rng.lognormal(mean=0.0, sigma=0.4, size=n_days)          # day-to-day noise
    return np.clip(vol, 1.0, None)

vol_by_series = {row.series: generate_synthetic_volume(row.thickness) for row in series.itertuples()}
vol_df = pd.DataFrame(vol_by_series)
vol_df.index.name = "trading_day"
vol_df.describe().T[["mean", "min", "max"]]

## 3. (Optional) real pull -- swap in here

Uncomment and adapt once you have outbound access to Kalshi. The wrapper is guarded so this cell is a no-op offline and the notebook still completes on synthetic data.

In [ ]:
USE_REAL_DATA = False  # flip to True in an environment with Kalshi API access

if USE_REAL_DATA:
    import kalshi_api
    real = {}
    for row in series.itertuples():
        # markets = kalshi_api.list_markets(series_ticker=row.ticker)
        # for each market pull candlesticks and read the 'volume' field per day,
        # then sum across the series' markets to get daily series volume:
        # candles = kalshi_api.get_market_candlesticks(row.ticker, mkt, start_ts, end_ts)
        # real[row.series] = candles.groupby('date')['volume'].sum()
        pass
    # vol_df = pd.DataFrame(real)  # <- replaces the synthetic vol_df above
    print("Wire up the real pull per the comments, then set vol_df = pd.DataFrame(real).")
else:
    print("Using synthetic volume (no network). Set USE_REAL_DATA=True with API access.")

## 4. Liquidity by series, common (log) scale

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for col in vol_df.columns:
    ax.plot(vol_df.index, vol_df[col], label=col, linewidth=1.3)
ax.set_yscale("log")
ax.set_xlabel("Trading day")
ax.set_ylabel("Daily contract volume (log scale)")
ax.set_title("SYNTHETIC DATA -- Kalshi macro series liquidity (replace via src/kalshi_api.py)")
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## 5. Trailing-volume ranking and an illustrative cost-to-move

Section 3's key quantity is *cost-to-move*: dollars needed to shift a thin market's implied probability by, say, 10 points in the pre-FOMC window. With real order-book depth this is estimable; here we show the relative ordering so the argument's structure is visible. A $7M position (Kalshi's per-market exposure cap, p.8) is a rounding error against the FFR series but can be most of total volume in a thin one.

In [ ]:
trailing = vol_df.tail(7).sum().sort_values()
cap = 7_000_000  # Kalshi max exposure per market, FEDS p.8 (dollars, not contracts -- illustrative overlay)

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.barh(trailing.index, trailing.values, color="tab:gray")
ax.axvline(cap, color="tab:red", linestyle="--", label="$7M per-market exposure cap (FEDS p.8)")
ax.set_xscale("log")
ax.set_xlabel("Trailing 7-day volume (log scale) with $7M cap overlaid")
ax.set_title("SYNTHETIC DATA -- where the $7M cap actually binds")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
trailing

## 6. What this grounds in Section 3

- The **relative** thinness of GDP / recession / core-CPI-annual vs. the FFR series (the chart in cell 4).
- Where the **$7M cap binds** vs. is irrelevant (cell 5): binding constraints only matter in the thin markets, which is exactly where manipulation risk concentrates.
- The seam for a real **cost-to-move** estimate once order-book depth is pulled.

Replace every `SYNTHETIC DATA` chart title with a real pull, then carry the resulting dollar figures into the `[DATA PLACEHOLDER]` slots in `paper/sections/3_manipulation_risk.md`.